# 🔗 Pipeline DAG Visualizer
Wizualizacja zależności procesów na podstawie konfiguracji z Apache Trino.

**Instalacja zależności (jednorazowo):**
```bash
pip install trino pyvis networkx pandas
```

In [ ]:
# ── Konfiguracja połączenia z Trino ───────────────────────────────────────────
TRINO_HOST     = "localhost"       # adres Trino coordinator
TRINO_PORT     = 8080
TRINO_USER     = "your_user"
TRINO_CATALOG  = "hive"            # katalog w Trino
TRINO_SCHEMA   = "pipeline_config" # schemat gdzie jest konfiguracja

# Czy używać prawdziwego Trino? False = załaduje przykładowe dane
USE_TRINO = False

In [ ]:
import pandas as pd
import networkx as nx
from pyvis.network import Network
from IPython.display import display, HTML
import json

print("✅ Zależności załadowane")

## 1. Połączenie z Trino i pobranie danych

Notebook oczekuje tabeli konfiguracyjnej w Trino z następującymi kolumnami:

| Kolumna | Typ | Opis |
|---|---|---|
| `from_id` | VARCHAR | ID węzła źródłowego (np. `raw.orders`) |
| `from_label` | VARCHAR | Wyświetlana nazwa |
| `from_type` | VARCHAR | `kafka` / `job` / `dwh` |
| `to_id` | VARCHAR | ID węzła docelowego |
| `to_label` | VARCHAR | Wyświetlana nazwa |
| `to_type` | VARCHAR | `kafka` / `job` / `dwh` |
| `edge_label` | VARCHAR | Etykieta krawędzi (opcjonalnie) |
| `active` | BOOLEAN | Czy zależność jest aktywna |
| `owner` | VARCHAR | Właściciel procesu (opcjonalnie) |
| `sla_minutes` | INTEGER | SLA w minutach (opcjonalnie) |

In [ ]:
# ── Zapytanie SQL – dostosuj do swojego schematu ──────────────────────────────
SQL_DEPENDENCIES = """
SELECT
    from_id,
    from_label,
    from_type,
    to_id,
    to_label,
    to_type,
    edge_label,
    active,
    owner,
    sla_minutes
FROM pipeline_config.dependencies
WHERE active = true
ORDER BY from_id, to_id
"""

# ── Alternatywne zapytanie: jeśli masz osobne tabele topics i jobs ────────────
SQL_DEPENDENCIES_JOINED = """
SELECT
    t.kafka_topic          AS from_id,
    t.kafka_topic          AS from_label,
    'kafka'                AS from_type,
    j.job_name             AS to_id,
    j.job_display_name     AS to_label,
    'job'                  AS to_type,
    'triggers'             AS edge_label,
    j.is_active            AS active,
    j.owner_team           AS owner,
    j.sla_minutes          AS sla_minutes
FROM kafka_triggers t
JOIN jobs j ON j.trigger_topic = t.kafka_topic

UNION ALL

SELECT
    j.job_name             AS from_id,
    j.job_display_name     AS from_label,
    'job'                  AS from_type,
    o.output_topic         AS to_id,
    o.output_topic         AS to_label,
    'kafka'                AS to_type,
    'publishes'            AS edge_label,
    j.is_active            AS active,
    j.owner_team           AS owner,
    NULL                   AS sla_minutes
FROM jobs j
JOIN job_outputs o ON o.job_name = j.job_name
"""

In [ ]:
def fetch_from_trino(sql: str) -> pd.DataFrame:
    """Pobiera dane z Trino i zwraca DataFrame."""
    import trino
    conn = trino.dbapi.connect(
        host=TRINO_HOST,
        port=TRINO_PORT,
        user=TRINO_USER,
        catalog=TRINO_CATALOG,
        schema=TRINO_SCHEMA,
    )
    df = pd.read_sql(sql, conn)
    conn.close()
    return df


def example_data() -> pd.DataFrame:
    """Przykładowe dane do testowania bez Trino."""
    rows = [
        ("raw.orders",      "raw.orders",       "kafka", "job.validate",     "Validate Orders",   "job",   "trigger",  True,  "team-data", 5),
        ("job.validate",    "Validate Orders",   "job",   "clean.orders",     "clean.orders",      "kafka", "publishes",True,  "team-data", None),
        ("raw.customers",   "raw.customers",     "kafka", "job.enrich",       "Enrich Customers",  "job",   "trigger",  True,  "team-crm",  None),
        ("job.enrich",      "Enrich Customers",  "job",   "clean.customers",  "clean.customers",   "kafka", "publishes",True,  "team-crm",  None),
        ("clean.orders",    "clean.orders",      "kafka", "job.join",         "Join & Aggregate",  "job",   "trigger",  True,  "team-data", 15),
        ("clean.customers", "clean.customers",   "kafka", "job.join",         "Join & Aggregate",  "job",   "trigger",  True,  "team-data", 15),
        ("job.join",        "Join & Aggregate",  "job",   "dwh.orders_full",  "orders_full",       "dwh",   "writes",   True,  "team-data", None),
        ("dwh.orders_full", "orders_full",       "dwh",   "job.report_daily", "Daily Report",      "job",   "triggers", True,  "team-bi",   None),
        ("dwh.orders_full", "orders_full",       "dwh",   "job.ml_features",  "ML Features",       "job",   "triggers", False, "team-ml",   None),
        ("job.report_daily","Daily Report",      "job",   "out.report",       "report.daily",      "kafka", "publishes",True,  "team-bi",   None),
        ("raw.inventory",   "raw.inventory",     "kafka", "job.inv_check",    "Inventory Check",   "job",   "trigger",  True,  "team-ops",  30),
        ("job.inv_check",   "Inventory Check",   "job",   "dwh.inventory",    "inventory",         "dwh",   "writes",   True,  "team-ops",  None),
        ("dwh.inventory",   "inventory",         "dwh",   "job.join",         "Join & Aggregate",  "job",   "triggers", True,  "team-data", None),
    ]
    return pd.DataFrame(rows, columns=[
        "from_id", "from_label", "from_type",
        "to_id",   "to_label",   "to_type",
        "edge_label", "active", "owner", "sla_minutes"
    ])


# ── Załaduj dane ──────────────────────────────────────────────────────────────
if USE_TRINO:
    df = fetch_from_trino(SQL_DEPENDENCIES)
    print(f"✅ Trino: pobrano {len(df)} zależności")
else:
    df = example_data()
    print(f"✅ Przykładowe dane: {len(df)} zależności")

display(df)

## 2. Budowanie grafu (NetworkX)

In [ ]:
# ── Kolory i kształty węzłów według typu ─────────────────────────────────────
NODE_STYLES = {
    "kafka": {"color": "#00e5ff", "shape": "hexagon",  "size": 22},
    "job":   {"color": "#b2ff59", "shape": "box",      "size": 20},
    "dwh":   {"color": "#ff4d6d", "shape": "database", "size": 24},
}
INACTIVE_COLOR = "#555555"


def build_graph(df: pd.DataFrame) -> nx.DiGraph:
    G = nx.DiGraph()

    for _, row in df.iterrows():
        if row.from_id not in G.nodes:
            G.add_node(row.from_id, label=row.from_label, node_type=row.from_type,
                       active=bool(row.active))
        if row.to_id not in G.nodes:
            G.add_node(row.to_id, label=row.to_label, node_type=row.to_type,
                       active=bool(row.active),
                       owner=row.get("owner", ""),
                       sla=row.get("sla_minutes", None))
        G.add_edge(row.from_id, row.to_id,
                   label=row.get("edge_label", "") or "",
                   active=bool(row.active))

    return G


G = build_graph(df)
print(f"Graf: {G.number_of_nodes()} węzłów, {G.number_of_edges()} krawędzi")
print(f"DAG (acykliczny): {nx.is_directed_acyclic_graph(G)}")

## 3. Wizualizacja DAG (pyvis)

In [ ]:
def render_dag(G: nx.DiGraph, output_file: str = "pipeline_dag.html",
               height: str = "700px") -> None:
    net = Network(
        height=height,
        width="100%",
        bgcolor="#0b0e14",
        font_color="#e0e6f0",
        directed=True,
        notebook=True,
        cdn_resources="in_line",  # wszystko wbudowane w HTML – działa offline!
    )

    net.set_options("""
    {
      "layout": {
        "hierarchical": {
          "enabled": true,
          "direction": "LR",
          "sortMethod": "directed",
          "levelSeparation": 200,
          "nodeSpacing": 90,
          "treeSpacing": 150
        }
      },
      "physics": { "enabled": false },
      "edges": {
        "arrows": { "to": { "enabled": true, "scaleFactor": 0.8 } },
        "smooth": { "type": "cubicBezier", "forceDirection": "horizontal" },
        "color": { "color": "#2a3550", "highlight": "#00e5ff" },
        "width": 1.5
      },
      "interaction": { "hover": true, "navigationButtons": true }
    }
    """)

    for node_id, attrs in G.nodes(data=True):
        node_type = attrs.get("node_type", "job")
        active    = attrs.get("active", True)
        style     = NODE_STYLES.get(node_type, NODE_STYLES["job"])
        color     = style["color"] if active else INACTIVE_COLOR
        owner     = attrs.get("owner", "")
        sla       = attrs.get("sla", None)

        type_labels = {"kafka": "Kafka Topic", "job": "Proces/Job", "dwh": "DWH Table"}
        tooltip = f"<b>{attrs.get('label', node_id)}</b><br>Typ: {type_labels.get(node_type, node_type)}<br>Status: {'✅ aktywny' if active else '⛔ nieaktywny'}"
        if owner: tooltip += f"<br>Owner: {owner}"
        if sla:   tooltip += f"<br>SLA: {sla} min"

        net.add_node(
            node_id,
            label=attrs.get("label", node_id),
            title=tooltip,
            shape=style["shape"],
            size=style["size"],
            color={"background": "#0d1018", "border": color,
                   "highlight": {"border": "#ffffff", "background": "#0d1018"}},
            font={"color": color, "size": 13, "face": "monospace"},
            borderWidth=2,
        )

    for src, dst, attrs in G.edges(data=True):
        active = attrs.get("active", True)
        net.add_edge(
            src, dst,
            title=attrs.get("label", ""),
            label=attrs.get("label", ""),
            color="#2a3550" if active else "#333",
            dashes=not active,
            font={"color": "#5a6480", "size": 10},
        )

    net.show(output_file)
    print(f"✅ Graf zapisany: {output_file}")


render_dag(G)

## 4. Analiza grafu (opcjonalnie)

In [ ]:
print("=" * 50)
print("ANALIZA GRAFU")
print("=" * 50)

sources = [n for n, d in G.in_degree() if d == 0]
print(f"\n📥 Źródła (brak poprzedników): {len(sources)}")
for s in sources:
    print(f"   • {G.nodes[s].get('label', s)}")

sinks = [n for n, d in G.out_degree() if d == 0]
print(f"\n📤 Końce pipeline (brak następników): {len(sinks)}")
for s in sinks:
    print(f"   • {G.nodes[s].get('label', s)}")

print(f"\n⚡ Top 5 węzłów (in-degree + out-degree):")
degree_df = pd.DataFrame([
    {"node": G.nodes[n].get("label", n),
     "in":  G.in_degree(n),
     "out": G.out_degree(n),
     "total": G.in_degree(n) + G.out_degree(n)}
    for n in G.nodes
]).sort_values("total", ascending=False).head(5)
display(degree_df)

if nx.is_directed_acyclic_graph(G):
    longest = nx.dag_longest_path(G)
    labels  = [G.nodes[n].get("label", n) for n in longest]
    print(f"\n🛤️  Najdłuższa ścieżka ({len(longest)} węzłów):")
    print("   " + " → ".join(labels))